# Exercise for Unit 4.1 Naïve Bayes

**Name:** Jethro A. Rendon  
**Year and Section:** BSCS 3-B AI  
**Date:** February 24, 2026

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [2]:
# Define the dataset
documents = [
    "Free money now!!!",
    "Hi mom, how are you?",
    "Lowest price for your meds",
    "Are we still on for dinner?",
    "Win a free iPhone today",
    "Let's catch up tomorrow at the office",
    "Meeting at 3 PM tomorrow",
    "Get 50% off, limited time!",
    "Team meeting in the office",
    "Click here for prizes!",
    "Can you send the report?"
]

labels = ['SPAM', 'HAM', 'SPAM', 'HAM', 'SPAM', 'HAM', 'HAM', 'SPAM', 'HAM', 'SPAM', 'HAM']

# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

## 1: Manual Naïve Bayes Implementation

### Task 1.1: Generate Bag of Words (Word Frequency)

In [3]:
def preprocess_text(text):
    """
    Preprocess text by converting to lowercase and removing punctuation.
    """
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation (simple approach)
    text = ''.join(char if char.isalnum() or char.isspace() else ' ' for char in text)
    # Split into tokens
    tokens = text.split()
    return tokens

def generate_bag_of_words(documents, labels):
    """
    Generate bag of words representation showing word frequencies per class.
    Returns:
        - vocabulary: set of all unique words
        - word_freq_by_class: dictionary with word frequencies for each class
        - doc_tokens: list of tokenized documents
    """
    vocabulary = set()
    word_freq_by_class = {'HAM': Counter(), 'SPAM': Counter()}
    doc_tokens = []
    
    for doc, label in zip(documents, labels):
        tokens = preprocess_text(doc)
        doc_tokens.append(tokens)
        vocabulary.update(tokens)
        word_freq_by_class[label].update(tokens)
    
    return vocabulary, word_freq_by_class, doc_tokens

# Generate bag of words
vocabulary, word_freq_by_class, doc_tokens = generate_bag_of_words(documents, labels)

print(f"\nVocabulary Size: {len(vocabulary)}")
print(f"Vocabulary: {sorted(vocabulary)}")

print("\nHAM Class:")
for word, freq in sorted(word_freq_by_class['HAM'].items()):
    print(f"  {word}: {freq}")
print("\nSPAM Class:")
for word, freq in sorted(word_freq_by_class['SPAM'].items()):
    print(f"  {word}: {freq}")


Vocabulary Size: 45
Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

HAM Class:
  3: 1
  are: 2
  at: 2
  can: 1
  catch: 1
  dinner: 1
  for: 1
  hi: 1
  how: 1
  in: 1
  let: 1
  meeting: 2
  mom: 1
  office: 2
  on: 1
  pm: 1
  report: 1
  s: 1
  send: 1
  still: 1
  team: 1
  the: 3
  tomorrow: 2
  up: 1
  we: 1
  you: 2

SPAM Class:
  50: 1
  a: 1
  click: 1
  for: 2
  free: 2
  get: 1
  here: 1
  iphone: 1
  limited: 1
  lowest: 1
  meds: 1
  money: 1
  now: 1
  off: 1
  price: 1
  prizes: 1
  time: 1
  today: 1
  win: 1
  your: 1


### Task 1.2: Calculate Prior Probabilities for HAM and SPAM

In [4]:
def calculate_priors(labels):
    """
    Calculate prior probabilities P(class) for each class.
    Prior = count of documents in class / total number of documents
    """
    total_docs = len(labels)
    class_counts = Counter(labels)
    priors = {}
    
    for class_label, count in class_counts.items():
        priors[class_label] = count / total_docs
    
    return priors, class_counts

# Calculate priors
priors, class_counts = calculate_priors(labels)


print(f"\nTotal documents: {len(labels)}")
print(f"HAM documents: {class_counts['HAM']}")
print(f"SPAM documents: {class_counts['SPAM']}")
print(f"\nP(HAM) = {class_counts['HAM']}/{len(labels)} = {priors['HAM']:.4f}")
print(f"P(SPAM) = {class_counts['SPAM']}/{len(labels)} = {priors['SPAM']:.4f}")


Total documents: 11
HAM documents: 6
SPAM documents: 5

P(HAM) = 6/11 = 0.5455
P(SPAM) = 5/11 = 0.4545


### Task 1.3: Calculate Likelihood of Tokens with Respect to Class

In [5]:
def calculate_likelihoods(vocabulary, word_freq_by_class, alpha=1):
    """
    Calculate likelihood P(word|class) for each word in vocabulary.
    Uses Laplace smoothing (add-alpha smoothing).
    
    P(word|class) = (count(word in class) + alpha) / (total words in class + alpha * |V|)
    
    Args:
        vocabulary: set of all unique words
        word_freq_by_class: word frequencies for each class
        alpha: smoothing parameter (default=1 for Laplace smoothing)
    """
    likelihoods = {'HAM': {}, 'SPAM': {}}
    vocab_size = len(vocabulary)
    
    for class_label in ['HAM', 'SPAM']:
        # Total word count in this class
        total_words = sum(word_freq_by_class[class_label].values())
        
        # Calculate likelihood for each word in vocabulary
        for word in vocabulary:
            word_count = word_freq_by_class[class_label].get(word, 0)
            likelihood = (word_count + alpha) / (total_words + alpha * vocab_size)
            likelihoods[class_label][word] = likelihood
    
    return likelihoods

# Calculate likelihoods with Laplace smoothing
likelihoods = calculate_likelihoods(vocabulary, word_freq_by_class, alpha=1)


print("\nUsing Laplace Smoothing (alpha = 1)")
print(f"Vocabulary size: {len(vocabulary)}")

# Calculate total words per class
total_ham = sum(word_freq_by_class['HAM'].values())
total_spam = sum(word_freq_by_class['SPAM'].values())
print(f"\nTotal words in HAM: {total_ham}")
print(f"Total words in SPAM: {total_spam}")

print("Likelihood P(word|HAM):")

for word in sorted(vocabulary)[:10]:  # Show first 10 for brevity
    count = word_freq_by_class['HAM'].get(word, 0)
    print(f"  P('{word}'|HAM) = ({count} + 1) / ({total_ham} + {len(vocabulary)}) = {likelihoods['HAM'][word]:.6f}")


print("Likelihood P(word|SPAM):")

for word in sorted(vocabulary)[:10]:  # Show first 10 for brevity
    count = word_freq_by_class['SPAM'].get(word, 0)
    print(f"  P('{word}'|SPAM) = ({count} + 1) / ({total_spam} + {len(vocabulary)}) = {likelihoods['SPAM'][word]:.6f}")


Using Laplace Smoothing (alpha = 1)
Vocabulary size: 45

Total words in HAM: 34
Total words in SPAM: 22
Likelihood P(word|HAM):
  P('3'|HAM) = (1 + 1) / (34 + 45) = 0.025316
  P('50'|HAM) = (0 + 1) / (34 + 45) = 0.012658
  P('a'|HAM) = (0 + 1) / (34 + 45) = 0.012658
  P('are'|HAM) = (2 + 1) / (34 + 45) = 0.037975
  P('at'|HAM) = (2 + 1) / (34 + 45) = 0.037975
  P('can'|HAM) = (1 + 1) / (34 + 45) = 0.025316
  P('catch'|HAM) = (1 + 1) / (34 + 45) = 0.025316
  P('click'|HAM) = (0 + 1) / (34 + 45) = 0.012658
  P('dinner'|HAM) = (1 + 1) / (34 + 45) = 0.025316
  P('for'|HAM) = (1 + 1) / (34 + 45) = 0.025316
Likelihood P(word|SPAM):
  P('3'|SPAM) = (0 + 1) / (22 + 45) = 0.014925
  P('50'|SPAM) = (1 + 1) / (22 + 45) = 0.029851
  P('a'|SPAM) = (1 + 1) / (22 + 45) = 0.029851
  P('are'|SPAM) = (0 + 1) / (22 + 45) = 0.014925
  P('at'|SPAM) = (0 + 1) / (22 + 45) = 0.014925
  P('can'|SPAM) = (0 + 1) / (22 + 45) = 0.014925
  P('catch'|SPAM) = (0 + 1) / (22 + 45) = 0.014925
  P('click'|SPAM) = (1 + 1

### Task 1.4: Classify Test Sentences Using Manual Implementation

In [6]:
def classify_naive_bayes(text, priors, likelihoods, vocabulary):
    """
    Classify a text using Naive Bayes.
    
    For each class:
        score = log(P(class)) + sum(log(P(word|class)) for each word in text)
    
    Returns the class with highest score.
    """
    tokens = preprocess_text(text)
    scores = {}
    details = {}
    
    for class_label in ['HAM', 'SPAM']:
        # Start with log prior
        score = math.log(priors[class_label])
        calculation_steps = [f"log(P({class_label})) = log({priors[class_label]:.4f}) = {score:.4f}"]
        
        # Add log likelihood for each word
        for word in tokens:
            if word in vocabulary:
                likelihood = likelihoods[class_label][word]
                score += math.log(likelihood)
                calculation_steps.append(f"  + log(P('{word}'|{class_label})) = log({likelihood:.6f}) = {math.log(likelihood):.4f}")
            else:
                # Word not in vocabulary - use smoothing
                calculation_steps.append(f"  + log(P('{word}'|{class_label})) = (unseen word, using smoothing)")
        
        scores[class_label] = score
        details[class_label] = calculation_steps
    
    # Return class with highest score
    predicted_class = max(scores, key=scores.get)
    return predicted_class, scores, details

for i, test_sentence in enumerate(test_sentences, 1):

    print(f"Test Sentence {i}: \"{test_sentence}\"")
  
    predicted_class, scores, details = classify_naive_bayes(
        test_sentence, priors, likelihoods, vocabulary
    )
    
    print(f"\nTokens: {preprocess_text(test_sentence)}")
    
    # Show calculation for HAM
   
    print("Calculation for HAM:")
    
    for step in details['HAM']:
        print(step)
    print(f"\nTotal Score (HAM): {scores['HAM']:.4f}")
    
    # Show calculation for SPAM
    print("Calculation for SPAM:")
    
    for step in details['SPAM']:
        print(step)
    print(f"\nTotal Score (SPAM): {scores['SPAM']:.4f}")
    
    # Show prediction

    print(f"PREDICTION: {predicted_class}")
   
    print(f"\nReasoning: {predicted_class} has higher score ({scores[predicted_class]:.4f} > {scores['SPAM' if predicted_class == 'HAM' else 'HAM']:.4f})")

Test Sentence 1: "Limited offer, click here!"

Tokens: ['limited', 'offer', 'click', 'here']
Calculation for HAM:
log(P(HAM)) = log(0.5455) = -0.6061
  + log(P('limited'|HAM)) = log(0.012658) = -4.3694
  + log(P('offer'|HAM)) = (unseen word, using smoothing)
  + log(P('click'|HAM)) = log(0.012658) = -4.3694
  + log(P('here'|HAM)) = log(0.012658) = -4.3694

Total Score (HAM): -13.7145
Calculation for SPAM:
log(P(SPAM)) = log(0.4545) = -0.7885
  + log(P('limited'|SPAM)) = log(0.029851) = -3.5115
  + log(P('offer'|SPAM)) = (unseen word, using smoothing)
  + log(P('click'|SPAM)) = log(0.029851) = -3.5115
  + log(P('here'|SPAM)) = log(0.029851) = -3.5115

Total Score (SPAM): -11.3231
PREDICTION: SPAM

Reasoning: SPAM has higher score (-11.3231 > -13.7145)
Test Sentence 2: "Meeting at 2 PM with the manager."

Tokens: ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']
Calculation for HAM:
log(P(HAM)) = log(0.5455) = -0.6061
  + log(P('meeting'|HAM)) = log(0.037975) = -3.2708
  + log(P('at

## 2: Using Scikit-Learn

### Task 2.1: Train Multinomial Naïve Bayes Classifier

In [7]:
# Create a pipeline with CountVectorizer and MultinomialNB
pipeline = Pipeline([
    ('vectorizer', CountVectorizer(lowercase=True, token_pattern=r'\b\w+\b')),
    ('classifier', MultinomialNB(alpha=1.0))  # alpha=1.0 for Laplace smoothing
])

# Train the model
pipeline.fit(documents, labels)


print("\nModel trained successfully!")
print(f"\nVectorizer vocabulary size: {len(pipeline.named_steps['vectorizer'].vocabulary_)}")
print(f"Class labels: {pipeline.named_steps['classifier'].classes_}")

# Show some model parameters
print(f"\nClass log priors:")
for class_label, log_prior in zip(pipeline.named_steps['classifier'].classes_, 
                                    pipeline.named_steps['classifier'].class_log_prior_):
    print(f"  {class_label}: {log_prior:.4f} (P = {np.exp(log_prior):.4f})")


Model trained successfully!

Vectorizer vocabulary size: 45
Class labels: ['HAM' 'SPAM']

Class log priors:
  HAM: -0.6061 (P = 0.5455)
  SPAM: -0.7885 (P = 0.4545)


### Task 2.2: Classify Test Sentences Using Scikit-Learn

In [8]:
# Predict classes
predictions = pipeline.predict(test_sentences)

# Get probability scores
probabilities = pipeline.predict_proba(test_sentences)

for i, (test_sentence, prediction, probs) in enumerate(zip(test_sentences, predictions, probabilities), 1):
    print(f"Test Sentence {i}: \"{test_sentence}\"")
    
    print(f"\nPredicted Class: {prediction}")
    print(f"\nClass Probabilities:")
    for class_label, prob in zip(pipeline.named_steps['classifier'].classes_, probs):
        print(f"  P({class_label}|text) = {prob:.6f} ({prob*100:.2f}%)")
    
    print(f"PREDICTION: {prediction}")

Test Sentence 1: "Limited offer, click here!"

Predicted Class: SPAM

Class Probabilities:
  P(HAM|text) = 0.083832 (8.38%)
  P(SPAM|text) = 0.916168 (91.62%)
PREDICTION: SPAM
Test Sentence 2: "Meeting at 2 PM with the manager."

Predicted Class: HAM

Class Probabilities:
  P(HAM|text) = 0.978118 (97.81%)
  P(SPAM|text) = 0.021882 (2.19%)
PREDICTION: HAM


### Comparison: Manual vs Scikit-Learn Results

In [ ]:
# Get manual predictions
manual_predictions = []
for test_sentence in test_sentences:
    predicted_class, _, _ = classify_naive_bayes(test_sentence, priors, likelihoods, vocabulary)
    manual_predictions.append(predicted_class)

# Get sklearn predictions
sklearn_predictions = pipeline.predict(test_sentences)

# Create comparison table
comparison_df = pd.DataFrame({
    'Test Sentence': test_sentences,
    'Manual Implementation': manual_predictions,
    'Scikit-Learn': sklearn_predictions,
    'Match': [m == s for m, s in zip(manual_predictions, sklearn_predictions)]
})

print("\n")
print(comparison_df.to_string(index=False))

print("CONCLUSION")
if all(comparison_df['Match']):
    print("\n Both implementations produce the same predictions!")
    print("  This validates our manual Naive Bayes implementation.")
else:
    print("\n There are differences between implementations.")
    print("  This may be due to differences in preprocessing or tokenization.")



                    Test Sentence Manual Implementation Scikit-Learn  Match
       Limited offer, click here!                  SPAM         SPAM   True
Meeting at 2 PM with the manager.                   HAM          HAM   True
CONCLUSION

 Both implementations produce the same predictions!
  This validates our manual Naive Bayes implementation.
